<a href="https://colab.research.google.com/github/weagan/Encoder-Decoder/blob/main/FLAN_T5_XXL_translation_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FLAN-T5-XXL (11B) Translation Notebook (Colab)

This notebook loads **Hugging Face `google/flan-t5-xxl` (11B)** and runs translation plus lightweight tests.

> **Runtime recommendation:** Colab GPU with high VRAM (A100 40GB preferred).
> This notebook pins the model to GPU (`device_map={'': 0}`) to avoid recent `bitsandbytes` auto-offload errors.

In [ ]:
# Verify runtime
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. FLAN-T5-XXL may be too large for CPU-only execution.')

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# Install dependencies (run once per fresh Colab runtime)
!pip -q install -U "transformers>=4.45" accelerate bitsandbytes sentencepiece sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

MODEL_ID = 'google/flan-t5-xxl'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Loading model (this can take a while)...')
# Use 4-bit quantization to further reduce memory usage.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    device_map={'': 0},
    quantization_config=bnb_config,
)

print('Model loaded.')

Loading tokenizer...


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/559 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def translate(text: str, source_lang: str, target_lang: str, max_new_tokens: int = 96) -> str:
    """Translate text with FLAN-T5 using an instruction prompt."""
    prompt = f"Translate from {source_lang} to {target_lang}: {text}"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
# Quick demo
sample = 'The weather is beautiful today.'
translated = translate(sample, 'English', 'French')
print('Source:', sample)
print('French:', translated)

Source: The weather is beautiful today.
French: Le temps est beau aujourd'hui.


## Lightweight Translation Tests

These are practical quality checks (not full benchmarking). They validate:
1. Output is non-empty.
2. Output differs from source text.
3. Basic corpus-level metric (`chrF`) is reported for a tiny test set.

In [ ]:
from sacrebleu.metrics import CHRF

TEST_CASES = [
    {'src': 'Hello, how are you?', 'ref': 'Bonjour, comment allez-vous ?'},
    {'src': 'I love reading books about science.', 'ref': "J'aime lire des livres sur la science."},
    {'src': 'Please turn left at the next street.', 'ref': 'Veuillez tourner à gauche à la prochaine rue.'},
]

preds, refs = [], []
for i, case in enumerate(TEST_CASES, start=1):
    pred = translate(case['src'], 'English', 'French')
    preds.append(pred)
    refs.append(case['ref'])

    print(f"[{i}] SRC: {case['src']}")
    print(f"    PRED: {pred}")
    print(f"    REF : {case['ref']}\n")

    assert isinstance(pred, str) and len(pred.strip()) > 0, 'Empty translation output'
    assert pred.strip().lower() != case['src'].strip().lower(), 'Output identical to source'

chrf = CHRF(word_order=2)
score = chrf.corpus_score(preds, [refs])
print(f"chrF++ score: {score.score:.2f}")

assert score.score > 25, f'chrF++ too low: {score.score:.2f}'
print('All tests passed.')

[1] SRC: Hello, how are you?
    PRED: Hello, comment allez-vous?
    REF : Bonjour, comment allez-vous ?

[2] SRC: I love reading books about science.
    PRED: J'aime lire les livres sur la science.
    REF : J'aime lire des livres sur la science.

[3] SRC: Please turn left at the next street.
    PRED: Veuillez prendre la gauche dans la prochaine rue.
    REF : Veuillez tourner à gauche à la prochaine rue.

chrF++ score: 71.22
All tests passed.


## Optional: Translate custom text

Edit `text`, `source_lang`, and `target_lang`.

In [ ]:
text = 'Machine learning can help translate languages efficiently.'
source_lang = 'English'
target_lang = 'Spanish'

print(translate(text, source_lang, target_lang))
#'El aprendizaje automático puede ayudar a traducir idiomas de manera eficiente.'


La aprendizaje de las máquinas puede ayudar a traducir idiomas eficientemente.


In [ ]:
text = 'Machine learning can help translate languages efficiently.'
source_lang = 'English'
target_lang = 'Irish'

print(translate(text, source_lang, target_lang))
#'Is féidir le foghlaim meaisín cabhrú le teangacha a aistriú go héifeachtúil.'
text = 'The dog did not find the bone.'
source_lang = 'English'
target_lang = 'Irish'

print(translate(text, source_lang, target_lang))
# Ní bhfuair an madra an cnámh


La aprendizaje de las máquinas puede ayudar a traducir idiomas eficientemente.
